# KATM vs KATMFast Speed Comparison

This notebook compares the runtime of the standard KATM implementation with KATMFast (S3 + S4 optimizations).

## Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

In [ ]:
from katm import KATM, KATMFast

## Load 20 Newsgroups Subset

In [ ]:
from sklearn.datasets import fetch_20newsgroups

# Load a larger subset for meaningful timing comparison
newsgroups = fetch_20newsgroups(
    remove=('headers', 'footers', 'quotes'),
    subset='train'
)

# Use 300 documents for this comparison
documents = newsgroups.data[:300]
print(f"Loaded {len(documents)} documents")

## Fit KATM (Standard Implementation)

In [ ]:
%%time

# Fit standard KATM
model = KATM(
    kp_algorithm='keybert',
    n_topics=10,
    n_keyphrases=10
)
model.fit(documents)

In [ ]:
print("KATM Topics:")
model.print_topics(n_words=5)

## Fit KATMFast (S3 + S4 Optimizations)

In [ ]:
%%time

# Fit KATMFast on the same data
model_fast = KATMFast(
    kp_algorithm='keybert',
    n_topics=10,
    n_keyphrases=10
)
model_fast.fit(documents)

In [ ]:
print("KATMFast Topics:")
model_fast.print_topics(n_words=5)

## Topic Output Comparison

Iterate and print topics side-by-side to see that outputs are nearly identical.

In [ ]:
print(f"{'='*60}")
print(f"{'KATM':^30} | {'KATMFast':^30}")
print(f"{'='*60}")

for topic_id in sorted(model.topics_.keys()):
    katm_words = [w for w, _ in model.topics_[topic_id][:5]]
    fast_words = [w for w, _ in model_fast.topics_[topic_id][:5]]
    print(f"Topic {topic_id}: {', '.join(katm_words):<30} | {', '.join(fast_words)}")

## Notes

- **KATMFast** uses S3 (vectorized anchor deduplication) and S4 (incremental MMR) optimizations.
- Outputs should be nearly identical between KATM and KATMFast.
- The speedup is more noticeable with larger datasets (more documents, more keyphrases).